最初に
- データセットは「1つの zip」にまとめる（中に images/ labels/ labels.txt を同梱）
  - DRIVE_DIR の中に置いた zip（配布データ）や、自分でアップロードした zip を使える
- 下のセル2で使うデータセット名（zip名から .zip を除いたもの）を選んでから、以降のセルを実行すること

In [ ]:
# 1. サンプルコードをダウンロード
%cd /content
!git clone https://github.com/wal-afk/yolo_sample
%cd yolo_sample
!git pull
%cd /content

In [ ]:
# 2. データセットの用意
from yolo_sample.util.dataset_combine import available, prepare, combine

# データセット置き場。
DRIVE_DIR = "dataset"
print("使えるデータセット:", available(DRIVE_DIR))

# 使うデータセットを名前で選ぶ
DATASETS = []        # 例: ["tc_b"] / ["ba_2_stop_complete", "mydata"]

sources = [prepare(name, DRIVE_DIR) for name in DATASETS]
assert sources, "DATASETS に使うデータセット名を指定してください（上の一覧から選ぶ）"
print("使うデータセット:", sources)

# データセットの結合
yaml_path, ds = combine(sources, work="dataset")
ds.print_labels_count()

In [ ]:
# 2(補足). データセットの画像をいくつか表示 
ds.show_images_for_each_label()

In [ ]:
# 3. 学習用のライブラリをインストール
%pip list | grep torch

%pip install -q 'ultralytics==8.3.162'

!yolo settings tensorboard=True > /dev/null
%load_ext tensorboard
%pip list | grep ultralytics

In [ ]:
# 4. 学習の実行
import ultralytics
from ultralytics import YOLO

ultralytics.checks()
model = YOLO('yolo11n.pt') # load a pretrained model
train_result = model.train(data='dataset/custom.yaml', exist_ok=True, epochs=50)

In [ ]:
# 5. 学習結果の評価
val_result=model.val(data='dataset/custom.yaml')

In [ ]:
# 6. 学習したモデルを使用した物体認識
from glob import glob
from yolo_sample.util.show_util import ImageViewer

confidential_threshhold = 0.25  # この値を大きくすると検出数が減る

results = model.predict(
    glob("dataset/images/val/**/*.png",recursive=True), conf=confidential_threshhold, verbose=False
)

viewer = ImageViewer([r.plot()[..., ::-1] for r in results])
viewer.display()

In [ ]:
# 参考：学習したモデルを使用した物体認識（数値表示）
def show_detail(results):
  for j,r in enumerate(results):
    boxes=r.boxes
    if len(boxes.cls)>0:
      print(f"image {j+1}/{len(results)}")
    for i in range(len(boxes.cls)):
      class_name=r.names[int(boxes.cls[i])]
      conf=float(boxes.conf[i])
      xywhn=boxes.xywhn[i].tolist()
      print(f"  {class_name} = {conf:.2f}")
      print(f"    x={xywhn[0]:.2f} y={xywhn[1]:.2f} w={xywhn[2]:.2f} h={xywhn[3]:.2f}")

show_detail(results)

In [ ]:
# 7.　学習済みモデルのダウンロード
from google.colab import files

files.download("runs/detect/train/weights/best.pt")

In [ ]:
# 参考: 学習済みモデル(汎用形式)の出力
%pip install -q 'onnxslim==0.1.57' 'onnx==1.18.0' 'onnxruntime-gpu' 'onnxscript'

path_onnx = model.export(format="onnx", opset=18)
!ls runs/detect/train/weights -l

In [ ]:
# 参考: 学習済みモデル(汎用形式)のダウンロード
files.download(path_onnx)

In [ ]:
# 参考：学習時の評価で使用された画像の表示
from yolo_sample.util.show_util import show_2_images_path
for path in glob("runs/detect/train/val_batch*_labels.jpg"):
  show_2_images_path(path, path[:-len("labels.jpg")]+"pred.jpg")

In [ ]:
# 参考：学習結果の確認ツールの起動
%tensorboard --logdir runs